# History-rung benchmark: GRU on the packet sequence

The neural sequence member of the history-rung benchmark. Where the other families receive engineered summaries of a link's earlier packets
(`12_History_Features`), the GRU receives the sequence itself: for every packet, the last `L = 30` packets of the same link, each described by
its deviation from the current anchor, the time gap to the packet before it, whether it used the same channel, its spreading factor and (from
rung G6) its SNR. The windows that reach further back than 30 packets (6 h, 24 h and 7 days, the 60-minute extremes and quantile, the 1 h and
24 h spreads and the packet count) enter as static inputs, so that the GRU sees as much history as the other families. Spreading factor and
channel are one-hot indicators, in the sequence and as static inputs, because their effect on the measured path loss is an offset per level.
Static inputs follow the rungs of notebook 13:

| rung | per-step inputs | static inputs |
|---|---|---|
| G1, link history | deviation, gap | the time-based windows |
| G4, + channel, clock, SF | + same channel, SF | + clock, SF and channel |
| G6, + earlier SNR, sensors | + SNR | + the five sensors, CO₂ above its 24 h minimum |
| G7, + link level and geometry | | + anchor, distance, walls |

Rows, folds, target (`y = PL - a`), hold-out, margin rule and the unseen-link test are those of notebook 13. The GRU width is screened on the
one-in-ten row sample; the chosen width is fitted with three seeds and averaged. Early stopping uses the last 10 % of the training rows in time order.


In [1]:
import json
import time
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
RANDOM_STATE = 42
TARGETS = (0.90, 0.95, 0.99)
L = 30
SEEDS = (0, 1, 2)


I0000 00:00:1791238678.176597  321895 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791238678.210371  321895 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1791238678.812410  321895 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


W0000 00:00:1791238679.316136  321895 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


In [2]:
train = pd.read_csv("Data_Files/history_train.csv")
test = pd.read_csv("Data_Files/history_test.csv")
d = pd.concat([train, test], ignore_index=True)
d["t"] = pd.to_datetime(d["t"], utc=True)
d = d.sort_values(["device_id", "t"]).reset_index(drop=True)   # per-link order for the sequences; time order is restored below

# spreading factor and channel as indicators: their effect on the measured path loss is an offset per level, not a trend
SF_LEVELS = (7, 8, 9, 10, 11, 12)
FREQS = sorted(d["frequency"].unique())
SF_COLS, CH_COLS = [f"SF{s}" for s in SF_LEVELS], [f"f{f}" for f in FREQS]
for s in SF_LEVELS:
    d[f"SF{s}"] = (d["SF"] == s).astype(np.float32)
for f in FREQS:
    d[f"f{f}"] = (d["frequency"] == f).astype(np.float32)

# per-step history, j packets back (j = 1 .. L), causal by construction
g = d.groupby("device_id", sort=False)
seq = {}
for j in range(1, L + 1):
    seq[("dev", j)] = (g["PL"].shift(j) - d["a"]).to_numpy(np.float32)
    seq[("gap", j)] = np.log1p(g["t"].diff(j).dt.total_seconds().to_numpy(np.float32)) if j == 1 else np.log1p((g["t"].shift(j - 1) - g["t"].shift(j)).dt.total_seconds().to_numpy(np.float32))
    seq[("same_channel", j)] = (g["frequency"].shift(j) == d["frequency"]).to_numpy(np.float32)
    for s in SF_LEVELS:
        seq[(f"SF{s}", j)] = (g["SF"].shift(j) == s).to_numpy(np.float32)
    seq[("snr", j)] = g["snr"].shift(j).to_numpy(np.float32)
STEP = {"G1": ["dev", "gap"], "G4": ["dev", "gap", "same_channel"] + SF_COLS, "G6": ["dev", "gap", "same_channel"] + SF_COLS + ["snr"]}
STEP["G7"] = STEP["G6"]
# the windows defined in time reach further back than 30 packets, so they enter as static inputs
TIME_WINDOWS = ["r_m6h", "r_m24h", "r_m7D", "r_max60", "r_min60", "r_q95", "s1h", "s24h", "n1h"]
STATIC = {"G1": TIME_WINDOWS, "G4": TIME_WINDOWS + ["hsin", "hcos", "dow", "work"] + SF_COLS + CH_COLS}
STATIC["G6"] = STATIC["G4"] + ["co2", "humidity", "pm25", "pressure", "temperature", "co2ex"]
STATIC["G7"] = STATIC["G6"] + ["a", "distance", "c_walls", "w_walls"]
seq_ok = np.all([~np.isnan(seq[("dev", j)]) for j in range(1, L + 1)], axis=0)

order = np.argsort(d["t"].to_numpy(), kind="stable")           # back to time order, same rows as notebooks 13 and 14
d = d.iloc[order].reset_index(drop=True)
seq = {k: v[order] for k, v in seq.items()}
seq_ok = seq_ok[order]
fold = d["fold"].to_numpy()
rungs = json.load(open("Data_Files/history_rungs.json"))
H7 = rungs["H7 + link level and geometry"]
ok = (d["y"].notna() & d[H7].notna().all(axis=1)).to_numpy() & seq_ok
is_test = fold == 9
cal = ok & (fold >= 0) & (fold < 9)
ev = ok & is_test
PL, a, y = d["PL"].to_numpy(), d["a"].to_numpy(), d["y"].to_numpy()
X_static = d[STATIC["G7"]].copy()
X_static = X_static.fillna(X_static[ok & (fold < 9)].median())
print({r: (len(STEP[r]), len(STATIC[r])) for r in STEP}, "(per-step inputs, static inputs)")
print(f"rows with a full {L}-packet history: training {int((ok & (fold < 9)).sum()):,} | calibration {int(cal.sum()):,} | hold-out {int(ev.sum()):,} of {int(is_test.sum()):,}")


{'G1': (2, 9), 'G4': (9, 27), 'G6': (10, 33), 'G7': (10, 37)} (per-step inputs, static inputs)
rows with a full 30-packet history: training 2,126,679 | calibration 1,773,285 | hold-out 531,983 of 532,055


In [3]:
def rmse(e):
    return float(np.sqrt(np.mean(np.square(e))))

def mixture_quantile(w, mu, sd, q):
    lo, hi = float((mu - 10 * sd).min()), float((mu + 10 * sd).max())
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if float((w * norm.cdf((mid - mu) / sd)).sum()) < q else (lo, mid)
    return 0.5 * (lo + hi)

def margins(e_cal, e_test):
    g = GaussianMixture(n_components=3, covariance_type="full", random_state=RANDOM_STATE).fit(e_cal.reshape(-1, 1))
    w, mu, sd = g.weights_, g.means_.ravel(), np.sqrt(g.covariances_.reshape(-1))
    out = {}
    for rho in TARGETS:
        emp = float(np.quantile(e_cal, rho))
        fm = max(emp, mixture_quantile(w, mu, sd, rho)) if rho >= 0.98 else emp
        out[f"margin {rho:.2f}"], out[f"reached {rho:.2f}"] = fm, float((e_test <= fm).mean())
    return out

def inputs(rung, rows):
    """(sequence array [n, L, steps], static array [n, statics]) for the given rows, oldest packet first."""
    S = np.stack([np.stack([seq[(name, j)][rows] for j in range(L, 0, -1)], axis=1) for name in STEP[rung]], axis=2)
    Z = X_static[STATIC[rung]].to_numpy(np.float32)[rows] if STATIC[rung] else np.zeros((len(rows), 0), np.float32)
    return S, Z

class GRU:
    def __init__(self, units, seeds=SEEDS):
        self.units, self.seeds, self.nets = units, seeds, []
    def fit(self, S, Z, y):
        self.mu, self.sd = S.reshape(-1, S.shape[2]).mean(0), S.reshape(-1, S.shape[2]).std(0) + 1e-6
        self.scaler = StandardScaler().fit(Z) if Z.shape[1] else None
        S, Z, y = (S - self.mu) / self.sd, (self.scaler.transform(Z).astype(np.float32) if Z.shape[1] else Z), y.astype(np.float32)
        n_val = max(1, len(y) // 10)
        self.nets = []
        for seed in self.seeds:
            tf.keras.utils.set_random_seed(seed)
            seq_in, sta_in = keras.Input(shape=S.shape[1:]), keras.Input(shape=(Z.shape[1],))
            h = keras.layers.GRU(self.units)(seq_in)
            h = keras.layers.Concatenate()([h, sta_in]) if Z.shape[1] else h
            h = keras.layers.Dense(32, activation="relu")(h)
            net = keras.Model([seq_in, sta_in], keras.layers.Dense(1)(h))
            net.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
            net.fit([S[:-n_val], Z[:-n_val]], y[:-n_val], validation_data=([S[-n_val:], Z[-n_val:]], y[-n_val:]), epochs=100, batch_size=4096, verbose=0,
                    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True), keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-5)])
            self.nets.append(net)
        return self
    def predict(self, S, Z):
        S, Z = (S - self.mu) / self.sd, (self.scaler.transform(Z).astype(np.float32) if Z.shape[1] else Z)
        return np.mean([net.predict([S, Z], batch_size=16384, verbose=0).ravel() for net in self.nets], axis=0)

def oof_and_test(make, rung):
    p = np.full(len(d), np.nan)
    for k in range(5):
        tr, va = np.flatnonzero(ok & (fold < k)), np.flatnonzero(ok & (fold == k))
        p[va] = make().fit(*inputs(rung, tr), y[tr]).predict(*inputs(rung, va))
    tr, te = np.flatnonzero(ok & (fold < 9)), np.flatnonzero(ev)
    model = make().fit(*inputs(rung, tr), y[tr])
    p[te] = model.predict(*inputs(rung, te))
    return p, model

def save_residuals(name, p):
    for split, mask in (("oof", cal), ("test", ev)):
        pd.DataFrame({"model": name, "split": split, "row_id": np.flatnonzero(mask), "fold": fold[mask], "time": d["t"].to_numpy()[mask],
                      "device_id": d["device_id"].to_numpy()[mask], "PL_true": PL[mask], "PL_pred": (a + p)[mask], "resid_db": (y - p)[mask]}
                     ).to_csv(f"Residuals/residuals_{name}_{split}.csv", index=False)

def evaluate(p):
    e = y - p
    row = {"hold-out RMSE": rmse(e[ev]), "hold-out MAE": float(np.abs(e[ev]).mean()), "OOF RMSE": rmse(e[cal])}
    row.update(margins(e[cal], e[ev]))
    return row


In [4]:
# Width screen on the one-in-ten sample (rung G4, one seed, the five folds)
sub = np.flatnonzero(ok & (fold < 9))[::10]
fold_sub = fold[sub]
screen = {}
for units in (16, 32, 64, 128):
    scores = []
    for k in range(5):
        tr, va = sub[fold_sub < k], sub[fold_sub == k]
        m = GRU(units, seeds=(0,)).fit(*inputs("G4", tr), y[tr])
        scores.append(rmse(y[va] - m.predict(*inputs("G4", va))))
    screen[units] = float(np.mean(scores))
    print(f"GRU({units}): sample CV RMSE {screen[units]:.3f} dB")
best_units = min(screen, key=screen.get)
json.dump({"units": best_units, "L": L, "screen": screen}, open("CV_Results/history_GRU_best_params.json", "w"), indent=1)
print("selected width:", best_units)


W0000 00:00:1791238694.023069  321895 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1791238694.119219  321895 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 29463 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5090, pci bus id: 0000:01:00.0, compute capability: 12.0a


I0000 00:00:1791238695.260670  322245 cuda_dnn.cc:461] Loaded cuDNN version 92400


GRU(16): sample CV RMSE 2.708 dB


GRU(32): sample CV RMSE 2.684 dB


GRU(64): sample CV RMSE 2.677 dB


GRU(128): sample CV RMSE 2.707 dB
selected width: 64


In [5]:
results = {}
for rung in ("G1", "G4", "G6", "G7"):
    t0 = time.time()
    p, _ = oof_and_test(lambda: GRU(best_units), rung)
    results[("GRU", rung.replace("G", "H"))] = evaluate(p)
    save_residuals(f"GRU_{rung.replace('G', 'H')}", p)
    print(f"GRU {rung}: hold-out RMSE {results[('GRU', rung.replace('G', 'H'))]['hold-out RMSE']:.3f} dB, margin 0.99 {results[('GRU', rung.replace('G', 'H'))]['margin 0.99']:.2f} dB  ({time.time() - t0:.0f} s)")
table = pd.DataFrame(results).T
table.index.names = ["family", "rung"]
table.to_csv("CV_Results/history_benchmark_GRU.csv")
table.round(3)


GRU G1: hold-out RMSE 2.527 dB, margin 0.99 9.17 dB  (2845 s)


GRU G4: hold-out RMSE 1.969 dB, margin 0.99 8.84 dB  (1660 s)


GRU G6: hold-out RMSE 1.901 dB, margin 0.99 9.05 dB  (1869 s)


GRU G7: hold-out RMSE 1.921 dB, margin 0.99 8.95 dB  (1829 s)


hold-out RMSE  hold-out MAE  OOF RMSE  margin 0.90  reached 0.90  \
family rung                                                                     
GRU    H1            2.527         1.911     2.794        2.717         0.867   
       H4            1.969         1.451     2.598        2.381         0.915   
       H6            1.901         1.391     2.538        2.195         0.898   
       H7            1.921         1.410     2.525        2.184         0.907   

             margin 0.95  reached 0.95  margin 0.99  reached 0.99  
family rung                                                        
GRU    H1          3.754         0.930        9.166         0.998  
       H4          3.275         0.964        8.842         0.999  
       H6          3.086         0.959        9.051         0.999  
       H7          3.076         0.963        8.950         0.999

In [6]:
# A link the model has never seen (rung G4)
dev = d["device_id"].to_numpy()
unseen = {}
for link in sorted(np.unique(dev)):
    other, this = dev != link, dev == link
    fit_m, cal_m, test_m = np.flatnonzero(ok & other & (fold >= 0) & (fold < 4)), np.flatnonzero(cal & other & (fold == 4)), np.flatnonzero(ev & this)
    m = GRU(best_units).fit(*inputs("G4", fit_m), y[fit_m])
    e_cal, e_test = y[cal_m] - m.predict(*inputs("G4", cal_m)), y[test_m] - m.predict(*inputs("G4", test_m))
    fm = float(np.quantile(e_cal, 0.99))
    unseen[("GRU", link)] = {"RMSE on the unseen link": rmse(e_test), "margin 0.99 from five links": fm, "reached on the unseen link": float((e_test <= fm).mean())}
unseen_df = pd.DataFrame(unseen).T
unseen_df.index.names = ["family", "link"]
unseen_df.to_csv("CV_Results/history_unseen_link_GRU.csv")
unseen_df.round(3)


RMSE on the unseen link  margin 0.99 from five links  \
family link                                                         
GRU    ED0                     1.990                        6.293   
       ED1                     2.132                        5.881   
       ED2                     2.192                        6.273   
       ED3                     1.864                        5.774   
       ED4                     1.890                        6.146   
       ED5                     1.900                        6.023   

             reached on the unseen link  
family link                              
GRU    ED0                        0.997  
       ED1                        0.997  
       ED2                        0.993  
       ED3                        0.998  
       ED4                        0.996  
       ED5                        0.997